In [115]:
# Source - https://stackoverflow.com/a/77503061
# Posted by Rajnish Dubey
# Retrieved 2026-05-07, License - CC BY-SA 4.0

import os, gzip, shutil
dir_name = r'../data'
def gz_extract(directory):
    extension = ".gz"
    os.chdir(directory)
    for item in os.listdir(directory): # loop through items in dir
      if item.endswith(extension): # check for ".gz" extension
          gz_name = os.path.abspath(item) # get full path of files
          file_name = (os.path.basename(gz_name)).rsplit('.',1)[0] #get file name for file within
          with gzip.open(gz_name,"rb") as f_in, open(file_name,"wb") as f_out:
              shutil.copyfileobj(f_in, f_out)
          os.remove(gz_name) # delete zipped file      
gz_extract(dir_name)


In [116]:
import re
def extract_gtf_attribute(series, key):
    return series.str.extract(fr'{key} "([^"]+)"', expand=False)

def get_appris_rank(attr):
    match = re.search(r'tag "appris_principal_(\d)"', attr)
    if match: return int(match.group(1))
    if 'tag "appris_principal"' in attr: return 1
    return 10  # Lowest priority

def format_interval(row):
    return f"{row['seqname']}:{row['start']}-{row['end']}({row['strand']})"

In [117]:
import pandas as pd
from pathlib import Path

gtf_path = Path('..') / 'data' / 'gencode.v49.primary_assembly.annotation.gtf'
rank_path = Path('..') / 'data' / 'gencode.v49.transcript_rankings.txt'

df = pd.read_csv(gtf_path, sep='\t', comment='#', header=None,
                 names=['seqname','source','feature','start','end','score','strand','frame','attribute'])

In [118]:
# helpers
def parse_intervals(interval_string):
    if pd.isna(interval_string) or interval_string == '':
        return []
    parsed = []
    for interval in interval_string.split(';'):
        interval = interval.strip()
        if not interval:
            continue
        chrom, rest = interval.split(':', 1)
        pos, strand = rest.split('(')
        start_s, end_s = pos.split('-')
        parsed.append((chrom, int(start_s), int(end_s), strand.rstrip(')')))
    return parsed


def merge_interval_list(intervals):
    """Merge overlapping/adjacent genomic intervals (same chrom and strand)."""
    if not intervals:
        return []

    intervals = sorted(intervals, key=lambda x: (x[0], x[3], x[1], x[2]))
    merged = []
    current = list(intervals[0])

    for chrom, start, end, strand in intervals[1:]:
        same_locus = chrom == current[0] and strand == current[3]
        if same_locus and start <= current[2] + 1:
            current[2] = max(current[2], end)
        else:
            merged.append(tuple(current))
            current = [chrom, start, end, strand]

    merged.append(tuple(current))
    return merged


def sort_intervals_in_transcript_direction(intervals):
    """Sort intervals in 5'->3' transcript order (not genomic order)."""
    if not intervals:
        return []
    strand = intervals[0][3]
    return sorted(intervals, key=lambda x: x[1], reverse=(strand == '-'))


def intervals_to_string(intervals):
    return '; '.join(f'{chrom}:{start}-{end}({strand})' for chrom, start, end, strand in intervals)


def merge_cds_with_codons_start_cds_stop(row):
    cds_intervals = parse_intervals(row.get('CDS', ''))
    start_intervals = parse_intervals(row.get('start_codon', ''))
    stop_intervals = parse_intervals(row.get('stop_codon', ''))

    merged = merge_interval_list(cds_intervals + start_intervals + stop_intervals)
    merged = sort_intervals_in_transcript_direction(merged)
    return intervals_to_string(merged)

def classify_utr(row, codon_map):
    tid = row['transcript_id']
    if row['feature'] == 'CDS':
        return 'CDS'
    if tid not in codon_map.index:
        return 'UTR_Incomplete'

    try:
        start_c_start = codon_map.loc[tid, ('start', 'start_codon')]
        stop_c_end = codon_map.loc[tid, ('end', 'stop_codon')]
    except KeyError:
        return 'UTR_Unknown'

    if row['strand'] == '+':
        if row['end'] <= start_c_start:
            return "5' UTR"
        if row['start'] >= stop_c_end:
            return "3' UTR"
    else:
        # Invert for - strand
        if row['start'] >= start_c_start:
            return "5' UTR"
        if row['end'] <= stop_c_end:
            return "3' UTR"
    return 'UTR_Internal'


In [119]:
# Extract gene_id and transcript_id
df['gene_id'] = extract_gtf_attribute(df['attribute'], 'gene_id')
df['transcript_id'] = extract_gtf_attribute(df['attribute'], 'transcript_id')
df['unfinished_mRNA'] = extract_gtf_attribute(df['attribute'], 'tag') == "mRNA_end_NF"
df['unfinished_CDS'] = extract_gtf_attribute(df['attribute'], 'tag') == "cds_end_NF"
df['nonsense_mediated_decay'] = extract_gtf_attribute(df['attribute'], 'transcript_type') == "nonsense_mediated_decay"

# Filter for ENST and Rank 1
ranks = pd.read_csv(rank_path, sep='\t', header=None, usecols=[1,3,4], names=['gene_id','rank','transcript_id'])
rank1_ensts = ranks.loc[(ranks['transcript_id'].str.startswith('ENST')) & (ranks['rank'] == 1), 'transcript_id'].unique()

# Select one transcript per gene based on APPRIS
transcripts = df[df['feature'] == 'transcript'].copy()
transcripts = transcripts[transcripts['transcript_id'].isin(rank1_ensts)].copy()
transcripts['appris_rank'] = transcripts['attribute'].apply(get_appris_rank)

final_best_ids = (
    transcripts.sort_values(['gene_id', 'appris_rank'])
    .drop_duplicates('gene_id')['transcript_id']
)

# Define start and stop codons
codons = df[
    (df['feature'].isin(['start_codon', 'stop_codon'])) &
    (df['transcript_id'].isin(final_best_ids))
].copy()
codons['interval'] = codons.apply(format_interval, axis=1)

codon_bounds = codons.groupby(['transcript_id', 'feature']).agg({'start': 'min', 'end': 'max'}).unstack()

# Classify UTR
feature_rows = df[
    (df['feature'].isin(['UTR', 'CDS'])) &
    (df['transcript_id'].isin(final_best_ids))
].copy()

feature_rows['feature'] = feature_rows.apply(lambda r: classify_utr(r, codon_bounds), axis=1)
feature_rows['interval'] = feature_rows.apply(format_interval, axis=1)

# Build coords including codons; merge codon coordinates into CDS but keep codon columns
final_features = ["5' UTR", 'start_codon', 'CDS', 'stop_codon', "3' UTR", 'appris_rank']
feature_table = feature_rows[feature_rows['feature'].isin(["5' UTR", 'CDS', "3' UTR"])][['gene_id', 'transcript_id', 'feature', 'interval']]
codon_table = codons[['gene_id', 'transcript_id', 'feature', 'interval']]

initialDB_coords_df = (
    pd.concat([feature_table, codon_table], ignore_index=True)
    .groupby(['gene_id', 'transcript_id', 'feature'], sort=False)['interval']
    .agg('; '.join)
    .unstack(fill_value='')
    .reindex(columns=final_features)
    .reset_index()
)

appris_map = transcripts.set_index('transcript_id')['appris_rank']
initialDB_coords_df['appris_rank'] = initialDB_coords_df['transcript_id'].map(appris_map).fillna(10).astype(int)

initialDB_coords_df['CDS'] = initialDB_coords_df.apply(merge_cds_with_codons_start_cds_stop, axis=1)

# Drop transcripts with unfinished CDS or mRNA after coords are built
coords_drop_unfinished = (
    initialDB_coords_df['transcript_id'].isin(df.loc[df['unfinished_CDS'] | df['unfinished_mRNA'], 'transcript_id'])
)

rows_dropped = int(coords_drop_unfinished.sum())
initialDB_coords_df = initialDB_coords_df.loc[~coords_drop_unfinished].copy()
print(f"Dropped {rows_dropped} rows with unfinished CDS or mRNA tags")

# #Drop nonsense mediated decay transcripts
# coords_drop_unfinished = (
#     initialDB_coords_df['transcript_id'].isin(df.loc[df['nonsense_mediated_decay'], 'transcript_id'])
# )
# rows_dropped = int(coords_drop_unfinished.sum())
# initialDB_coords_df = initialDB_coords_df.loc[~coords_drop_unfinished].copy()
# print(f"Dropped {rows_dropped} rows with nonsense mediated decay tags")

Dropped 309 rows with unfinished CDS or mRNA tags


In [120]:
appris_counts = initialDB_coords_df['appris_rank'].value_counts().sort_index()
appris_counts 

appris_rank
1     11621
2      1796
3      3473
4      1299
5       132
10     1903
Name: count, dtype: int64

In [121]:
initialDB_coords_df.to_csv('../data/initialDB_coords.csv', index=False)

In [122]:
assembly_path = Path('..') / 'data' / 'GRCh38.primary_assembly.genome.fa'

def load_genome_fasta(path):
    genome = {}
    with open(path) as f:
        chrom = None
        seq_chunks = []
        for line in f:
            if line.startswith('>'):
                if chrom: genome[chrom] = ''.join(seq_chunks)
                chrom = line[1:].split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        if chrom: genome[chrom] = ''.join(seq_chunks)
    return genome

genome = load_genome_fasta(assembly_path)

In [123]:
interval_pattern = re.compile(r'^(?P<chrom>[^:]+):(?P<start>\d+)-(?P<end>\d+)\((?P<strand>[+-])\)$')


def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    return seq.translate(str.maketrans('ACGTacgtNn', 'TGCAtgcaNn'))[::-1]


def get_chrom_sequence(chrom):
    """Return the sequence for a chromosome, trying both 'chr' and non-'chr' versions."""
    if chrom in genome:
        return genome[chrom]
    alt_chrom = chrom[3:] if chrom.startswith('chr') else f'chr{chrom}'
    if alt_chrom in genome:
        return genome[alt_chrom]
    raise KeyError(f'Chromosome {chrom!r} not found in loaded genome')


def interval_to_sequence(interval):
    """Return the sequence for a given interval."""
    match = interval_pattern.match(interval)
    if match is None:
        raise ValueError(f'Could not parse interval {interval!r}')

    chrom = match.group('chrom')
    start = int(match.group('start'))
    end = int(match.group('end'))
    strand = match.group('strand')

    sequence = get_chrom_sequence(chrom)[start - 1:end]
    if strand == '-':
        sequence = reverse_complement(sequence)
    return sequence


def intervals_to_sequence(interval_string):
    """Return the sequences of intervals indicated by the coordinates. The coordinates are concatenated as the separation
    before was only due to the fact they were on different exons.
    """
    if pd.isna(interval_string) or interval_string == '':
        return ''
    return ''.join(
        interval_to_sequence(interval.strip())
        for interval in interval_string.split(';')
        if interval.strip()
    )


sequence_cols = ["5' UTR", 'start_codon', 'CDS', 'stop_codon', "3' UTR"]
initialDB_df = initialDB_coords_df.copy()
for col in sequence_cols:
    initialDB_df[col] = initialDB_df[col].apply(intervals_to_sequence)

# Ensure empty entries become empty string
for col in sequence_cols:
    initialDB_df[col] = initialDB_df[col].fillna('')

# Add length columns only for UTR/CDS (not start/stop codons)
length_cols = ["5' UTR", 'CDS', "3' UTR"]
for col in length_cols:
    initialDB_df[f"{col}_len"] = initialDB_df[col].apply(len)

# initialDB_df.to_csv('../data/initialDB_sequences.csv', index=False)
initialDB_df.head()

feature,gene_id,transcript_id,5' UTR,start_codon,CDS,stop_codon,3' UTR,appris_rank,5' UTR_len,CDS_len,3' UTR_len
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATG,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,TAG,,1,60,981,0
1,ENSG00000284733.2,ENST00000426406.4,,ATG,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,TAA,1,0,939,3
2,ENSG00000284662.2,ENST00000332831.5,,ATG,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,TAA,1,0,939,3
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATG,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,TGA,,10,509,2535,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATG,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGA,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,3,16,2250,494


In [124]:
len(initialDB_df)

20224

In [125]:
not3 = initialDB_df[initialDB_df['CDS_len'] % 3 != 0].copy().drop(columns=["5' UTR", "3' UTR", "5' UTR_len", "3' UTR_len"])

not3['nonsense'] = not3['transcript_id'].isin(df.loc[df['nonsense_mediated_decay'], 'transcript_id'])

not3.head()

feature,gene_id,transcript_id,start_codon,CDS,stop_codon,appris_rank,CDS_len,nonsense
736,ENSG00000284686.1,ENST00000642129.1,,GAACCCCTACGTGGCAGCACTCTATAAGCAAGTGGGCTGCTTCCTC...,TGA,5,673,True
1099,ENSG00000289565.1,ENST00000632040.1,,CTGTTGAAGGCTGGATTCTCTTTGTAACTGGAGTCCATGAGGAAGC...,TGA,1,320,True
1365,ENSG00000160818.17,ENST00000368232.9,ATG,ATGAATGTCACCCCAGAGGTCAAGAGTCGTGGGATGAAGTTTGCTG...,,5,1339,False
1418,ENSG00000196266.6,ENST00000649616.1,ATG,ATGCCAAAGCTAAATTCCACTTTTGTGACTGAGTTCCTCTTTGAAG...,,1,940,False
1419,ENSG00000249730.1,ENST00000504970.1,ATG,ATGCCAAGGCCCAATTTCATGGCTGTGACAGAGTTTACATTTGAGG...,,1,935,False


In [126]:
# For now, drop all sequences in not3 -> revisit later if needed

initialDB_df = initialDB_df[initialDB_df['CDS_len'] % 3 == 0].copy()
initialDB_df.to_csv('../data/initialDB_sequences.csv', index=False)

len(initialDB_df)

19988

In [ ]:
RiboNN_df = initialDB_df[(initialDB_df['start_codon'] != '') & (initialDB_df['stop_codon'] != '')]
RiboNN_df.drop(['gene_id', 'start_codon', 'stop_codon', 'appris_rank', '5\' UTR_len', 'CDS_len', '3\' UTR_len'], axis=1, inplace=True)

RiboNN_df.to_csv('../data/RiboNN_sequences.csv', index=False)

RiboNN_df

feature,transcript_id,5' UTR,CDS,3' UTR
0,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,
1,ENST00000426406.4,,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA
2,ENST00000332831.5,,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA
3,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,
4,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...
...,...,...,...,...
20524,ENST00000622674.1,,ATGAAGATAGCAAACAACACAGTAGTGACAGAATTTATCCTCCTTG...,
20525,ENST00000616638.1,GAGAGACTACAACTCCCAGTATGCACCGCG,ATGCGCGCCTCACCCTGCATCTCCCAGCCCGCAGCCAGCTGGCATC...,
20526,ENST00000621028.1,GGAGAGACTACAACTCCCAGTATGCACCGCG,ATGCGCGCCTCACCCTGCATCTCCCAGCCCGCAGCCAGCTGGCATC...,TAAGTACTGCACGACTACTCGCTTCTGAAACTGATAGACACTGCCT...
20531,ENST00000613204.1,CGCGAGGCGCGCCGCGATCGGGGACTGTCCTAAGACGGGCGGGGCG...,ATGGAGCGCTACGCGGGCGCCTTGGAGGAGGTGGCGGACGGTGCCC...,


: 

In [127]:
print("Total non 3 CDS:", len(not3))

print("Missing both start and stop codon:", len(not3[(not3['start_codon'] == '') & (not3['stop_codon'] == '')]))

print("Missing start codon:", len(not3[not3['start_codon'] == '']))

print("Missing stop codon:", len(not3[not3['stop_codon'] == '']))

print("Missing neither codon:", len(not3[(not3['start_codon'] != '') & (not3['stop_codon'] != '')]))

not3[not3['start_codon'] == '']

Total non 3 CDS: 236
Missing both start and stop codon: 94
Missing start codon: 202
Missing stop codon: 128
Missing neither codon: 0


feature,gene_id,transcript_id,start_codon,CDS,stop_codon,appris_rank,CDS_len,nonsense
736,ENSG00000284686.1,ENST00000642129.1,,GAACCCCTACGTGGCAGCACTCTATAAGCAAGTGGGCTGCTTCCTC...,TGA,5,673,True
1099,ENSG00000289565.1,ENST00000632040.1,,CTGTTGAAGGCTGGATTCTCTTTGTAACTGGAGTCCATGAGGAAGC...,TGA,1,320,True
1442,ENSG00000258465.8,ENST00000485079.1,,GAAACACTAAGTGGATTAGCCAAAAATGCCACTGACCTTCAGAACT...,,5,914,False
2314,ENSG00000273269.3,ENST00000422269.1,,AGTCCGAGTGGAGAGAGCGAGCTGAGTGGTTGTGTGGTCGCGTCTC...,TGA,1,131,True
2538,ENSG00000211592.8,ENST00000390237.2,,GAACTGTGGCTGCACCATCTGTCTTCATCTTCCCGCCATCTGATGA...,TAG,1,323,False
...,...,...,...,...,...,...,...,...
20183,ENSG00000278646.1,ENST00000613352.1,,AAGTCACTAAATCTCAGCCCGAAAAGTGGGATGAAGAGGCCCAAGA...,TAG,1,125,False
20378,ENSG00000284987.1,ENST00000646191.1,,AGAGCCATGCACCTCACCTGAGGCCATGGGACCCTCTGGACACAGA...,TGA,5,163,True
20496,ENSG00000198888.2,ENST00000361390.2,,ATACCCATGGCCAACCTCCTACTCCTCATTGTACCCATTCTAATCG...,,1,956,False
20497,ENSG00000198763.3,ENST00000361453.3,,ATTAATCCCCTGGCCCAACCCGTCATCTACTCTACCATCTTTGCAG...,,1,1042,False


In [128]:
not_3 = not3.drop(columns=['appris_rank', 'CDS_len', 'start_codon', 'stop_codon'])

not_3.head()

feature,gene_id,transcript_id,CDS,nonsense
736,ENSG00000284686.1,ENST00000642129.1,GAACCCCTACGTGGCAGCACTCTATAAGCAAGTGGGCTGCTTCCTC...,True
1099,ENSG00000289565.1,ENST00000632040.1,CTGTTGAAGGCTGGATTCTCTTTGTAACTGGAGTCCATGAGGAAGC...,True
1365,ENSG00000160818.17,ENST00000368232.9,ATGAATGTCACCCCAGAGGTCAAGAGTCGTGGGATGAAGTTTGCTG...,False
1418,ENSG00000196266.6,ENST00000649616.1,ATGCCAAAGCTAAATTCCACTTTTGTGACTGAGTTCCTCTTTGAAG...,False
1419,ENSG00000249730.1,ENST00000504970.1,ATGCCAAGGCCCAATTTCATGGCTGTGACAGAGTTTACATTTGAGG...,False


In [129]:
codons = {
"TTT" : "F", "CTT" : "L", "ATT" : "I", "GTT" : "V", "TTC" : "F",
"CTC" : "L", "ATC" : "I", "GTC" : "V", "TTA" : "L", "CTA" : "L",
"ATA" : "I", "GTA" : "V", "TTG" : "L", "CTG" : "L", "ATG" : "M",
"GTG" : "V", "TCT" : "S", "CCT" : "P", "ACT" : "T", "GCT" : "A",
"TCC" : "S", "CCC" : "P", "ACC" : "T", "GCC" : "A", "TCA" : "S",
"CCA" : "P", "ACA" : "T", "GCA" : "A", "TCG" : "S", "CCG" : "P",
"ACG" : "T", "GCG" : "A", "TAT" : "Y", "CAT" : "H", "AAT" : "N",
"GAT" : "D", "TAC" : "Y", "CAC" : "H", "AAC" : "N", "GAC" : "D",
"TAA" : "Stop", "CAA" : "Q", "AAA" : "K", "GAA" : "E", "TAG" : "Stop", 
"CAG" : "Q", "AAG" : "K", "GAG" : "E", "TGT" : "C", "CGT" : "R",
"AGT" : "S", "GGT" : "G", "TGC" : "C", "CGC" : "R", "AGC" : "S",
"GGC" : "G", "TGA" : "Stop", "CGA" : "R", "AGA" : "R", "GGA" : "G",
"TGG" : "W", "CGG" : "R", "AGG" : "R", "GGG" : "G"
}

not_3['AA Seq'] = not_3['CDS'].apply(lambda seq: ''.join(codons.get(seq[i:i+3], 'X') for i in range(0, len(seq)-2, 3)))

not_3['Stop_Count'] = not_3['AA Seq'].apply(lambda aa_seq: aa_seq.count('Stop'))

not_3

feature,gene_id,transcript_id,CDS,nonsense,AA Seq,Stop_Count
736,ENSG00000284686.1,ENST00000642129.1,GAACCCCTACGTGGCAGCACTCTATAAGCAAGTGGGCTGCTTCCTC...,True,EPLRGSTLStopASGLLPLWLCHQPVFHRHCQSVHRAPASSLLECL...,9
1099,ENSG00000289565.1,ENST00000632040.1,CTGTTGAAGGCTGGATTCTCTTTGTAACTGGAGTCCATGAGGAAGC...,True,LLKAGFSLStopLESMRKPPKKTYTTNSQNMGKLKTFISTSTGEQD...,4
1365,ENSG00000160818.17,ENST00000368232.9,ATGAATGTCACCCCAGAGGTCAAGAGTCGTGGGATGAAGTTTGCTG...,False,MNVTPEVKSRGMKFAEEQLLKHGWTQGKGLGRKENGITQALRVTLK...,4
1418,ENSG00000196266.6,ENST00000649616.1,ATGCCAAAGCTAAATTCCACTTTTGTGACTGAGTTCCTCTTTGAAG...,False,MPKLNSTFVTEFLFEGFSSFRRQHKLVFFVVFLTLYLLTLSGNVII...,0
1419,ENSG00000249730.1,ENST00000504970.1,ATGCCAAGGCCCAATTTCATGGCTGTGACAGAGTTTACATTTGAGG...,False,MPRPNFMAVTEFTFEGFSIFEWHHRLILFVIFLVLYVLTLASNAII...,6
...,...,...,...,...,...,...
20497,ENSG00000198763.3,ENST00000361453.3,ATTAATCCCCTGGCCCAACCCGTCATCTACTCTACCATCTTTGCAG...,False,INPLAQPVIYSTIFAGTLITALSSHStopFFTStopVGLEINMLAF...,10
20502,ENSG00000198938.2,ENST00000362079.2,ATGACCCACCAATCACATGCCTATCATATAGTAAAACCCAGCCCAT...,False,MTHQSHAYHIVKPSPStopPLTGALSALLMTSGLAMStopFHFHSI...,9
20503,ENSG00000198840.2,ENST00000361227.2,ATAAACTTCGCCTTAATTTTAATAATCAACACCCTCCTAGCCTTAC...,False,INFALILIINTLLALLLIIITFStopLPQLNGYIEKSTPYECGFDP...,4
20505,ENSG00000198886.2,ENST00000361381.2,ATGCTAAAACTAATCGTCCCAACAATTATATTACTACCACTGACAT...,False,MLKLIVPTIILLPLTStopLSKKHIIStopINTTTHSLIISIIPLL...,12


In [130]:
def remove_stop(aa_seq):
    return aa_seq.replace('Stop', '')

blast_df = not_3[['gene_id', 'transcript_id', 'AA Seq']].copy()
blast_df['AA Seq'] = blast_df['AA Seq'].apply(remove_stop)

blast_df.to_csv('../data/blast.csv', index=False)

blast_df

feature,gene_id,transcript_id,AA Seq
736,ENSG00000284686.1,ENST00000642129.1,EPLRGSTLASGLLPLWLCHQPVFHRHCQSVHRAPASSLLECLQPFQ...
1099,ENSG00000289565.1,ENST00000632040.1,LLKAGFSLLESMRKPPKKTYTTNSQNMGKLKTFISTSTGEQDIRGI...
1365,ENSG00000160818.17,ENST00000368232.9,MNVTPEVKSRGMKFAEEQLLKHGWTQGKGLGRKENGITQALRVTLK...
1418,ENSG00000196266.6,ENST00000649616.1,MPKLNSTFVTEFLFEGFSSFRRQHKLVFFVVFLTLYLLTLSGNVII...
1419,ENSG00000249730.1,ENST00000504970.1,MPRPNFMAVTEFTFEGFSIFEWHHRLILFVIFLVLYVLTLASNAII...
...,...,...,...
20497,ENSG00000198763.3,ENST00000361453.3,INPLAQPVIYSTIFAGTLITALSSHFFTVGLEINMLAFIPVLTKKI...
20502,ENSG00000198938.2,ENST00000362079.2,MTHQSHAYHIVKPSPPLTGALSALLMTSGLAMFHFHSITLLILGLL...
20503,ENSG00000198840.2,ENST00000361227.2,INFALILIINTLLALLLIIITFLPQLNGYIEKSTPYECGFDPISPA...
20505,ENSG00000198886.2,ENST00000361381.2,MLKLIVPTIILLPLTLSKKHIIINTTTHSLIISIIPLLFFNQINNN...


In [131]:
# Load gencode protein translations keyed by protein ID
gencode_fasta_path = Path('..') / 'data' / 'gencode.v49.pc_translations.fa'

def load_protein_fasta_by_protein(fasta_path):
    """Load FASTA and map protein ID to amino acid sequence."""
    protein_dict = {}
    with open(fasta_path) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id is not None:
                    protein_dict[current_id] = ''.join(current_seq)
                header_parts = line[1:].split('|')
                current_id = header_parts[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            protein_dict[current_id] = ''.join(current_seq)
    return protein_dict

gencode_proteins = load_protein_fasta_by_protein(gencode_fasta_path)
print(f"Loaded {len(gencode_proteins)} protein sequences from gencode")
print(f"Sample keys: {list(gencode_proteins.keys())[:3]}")

Loaded 245535 protein sequences from gencode
Sample keys: ['ENSP00000493376.2', 'ENSP00000409316.1', 'ENSP00000329982.2']


In [132]:
# Reload GTF to extract protein_id mapping
gtf_path = Path('..') / 'data' / 'gencode.v49.primary_assembly.annotation.gtf'
gtf_full = pd.read_csv(gtf_path, sep='\t', comment='#', header=None,
                       names=['seqname','source','feature','start','end','score','strand','frame','attribute'])

protein_id_map = {}

# Extract protein_id from transcript records
for idx, row in gtf_full[gtf_full['feature'] == 'transcript'].iterrows():
    tid = extract_gtf_attribute(pd.Series([row['attribute']]), 'transcript_id').values[0]
    pid = extract_gtf_attribute(pd.Series([row['attribute']]), 'protein_id').values[0]
    if pd.notna(tid) and pd.notna(pid):
        protein_id_map[tid] = pid

print(f"Built mapping for {len(protein_id_map)} transcripts to protein IDs")
print(f"Sample mappings: {list(protein_id_map.items())[:3]}")

# Now create the matches_gencode_aa column using protein IDs
def matches_gencode(row, protein_map, gencode_proteins):
    """Check if AA Seq matches the gencode protein sequence"""
    tid = row['transcript_id']
    aa_seq = row['AA Seq']
    
    if pd.isna(aa_seq) or aa_seq == '':
        return False
    
    if tid not in protein_map:
        return False
    
    protein_id = protein_map[tid]
    if protein_id not in gencode_proteins:
        return False
    
    gencode_seq = gencode_proteins[protein_id]
    return aa_seq == gencode_seq

blast_df['matches_gencode_aa'] = blast_df.apply(
    lambda row: matches_gencode(row, protein_id_map, gencode_proteins),
    axis=1,
)

print(f"\nMatches found: {blast_df['matches_gencode_aa'].sum()}/{len(blast_df)}")
print(blast_df[['gene_id', 'transcript_id', 'matches_gencode_aa']].head(10))

Built mapping for 234024 transcripts to protein IDs
Sample mappings: [('ENST00000641515.2', 'ENSP00000493376.2'), ('ENST00000426406.4', 'ENSP00000409316.1'), ('ENST00000332831.5', 'ENSP00000329982.2')]

Matches found: 50/236
feature             gene_id      transcript_id  matches_gencode_aa
736       ENSG00000284686.1  ENST00000642129.1               False
1099      ENSG00000289565.1  ENST00000632040.1               False
1365     ENSG00000160818.17  ENST00000368232.9               False
1418      ENSG00000196266.6  ENST00000649616.1                True
1419      ENSG00000249730.1  ENST00000504970.1               False
1442      ENSG00000258465.8  ENST00000485079.1                True
2048      ENSG00000227152.6  ENST00000641557.1               False
2305      ENSG00000284701.1  ENST00000434431.2                True
2314      ENSG00000273269.3  ENST00000422269.1               False
2538      ENSG00000211592.8  ENST00000390237.2               False


In [133]:
matches = blast_df[blast_df['matches_gencode_aa']]

matches

feature,gene_id,transcript_id,AA Seq,matches_gencode_aa
1418,ENSG00000196266.6,ENST00000649616.1,MPKLNSTFVTEFLFEGFSSFRRQHKLVFFVVFLTLYLLTLSGNVII...,True
1442,ENSG00000258465.8,ENST00000485079.1,ETLSGLAKNATDLQNSSMSEEELTKAMEGLGMDEGDGEGNILPIMQ...,True
2305,ENSG00000284701.1,ENST00000434431.2,MAAEDREMMEARGAGESCPTFPKMVPGDSKSEGKPRAYLEAESQKP...,True
2540,ENSG00000211594.2,ENST00000390239.2,LTFGGGTKVEIK,True
6894,ENSG00000281613.2,ENST00000587816.2,MAKRLQAELSCPVCLDFFSCSISLSCTHVFCFDCIQRYILENHDFR...,True
7374,ENSG00000211691.2,ENST00000390338.2,GQELGKKIKVFGPGTKLIIT,True
8036,ENSG00000292271.1,ENST00000710614.1,GTSGR,True
8040,ENSG00000211767.1,ENST00000390415.1,STDTQYFGPGTRLTVL,True
11597,ENSG00000285509.1,ENST00000645041.1,YSPENFPYRRGPGMGVHVPATPQGSPMKDRLNLPSVLVLNSCGITC...,True
12311,ENSG00000257921.6,ENST00000546504.1,VDFRGKKVIELGAGTGIVGILAALQGWVSSASVAPAKAHILCWAPS...,True
